# TradeSense · Component 03 · notebook 53: historical similar windows, second version (protocol H1 v4; CPU, minutes)

**Why a v4.** Notebook 50 (H1 v3) found that retrieval of similar past windows works as *context* (H-H1a, H-H3, H-H4 hold) but two claims failed: same-family precision was 0.247 vs 0.25 (feature deviations added nothing over scores) and neighbours did not forecast the next-hour peak better than "the current peak". Those results stay reported as failed. This notebook tests a *new version*: richer window descriptors chosen on development data only (CAL-3), then confirmed once on held-out TEST-3 injected events and genuine TEST-B hours, with an honest "incremental value" question instead of a forecasting claim, plus a product question ("when most similar past windows were followed by an Alert, is an Alert more likely next hour?").

No model is retrained or changed. TEST-3 / TEST-B were opened once in T0, so these are **new pre-registered analyses on saved scores**.

**Protocol text hashed by this notebook:**


```
# Protocol H1 v4 - historical similar-window retrieval, second version (novelty 4)
Written BEFORE any v4 result exists. Supersedes nothing: the results of H1 v3 (notebook 50: H-H1a holds, H-H1b FAILS 0.247 vs 0.25, H-H2 FAILS 0.086 vs 0.20, H-H3 holds, H-H4 holds) stay as they are and stay reported as failed. v4 is a NEW version with a changed descriptor and a changed set of claims. Nothing frozen is changed (models, 6 features, normalisation, fusion stacker, thresholds q, Layer 3 sets).

## What v3 showed, and what v4 changes
1. Feature deviations added nothing over score summaries, and "max |z|" throws away the sign and the shape of a pattern. v4 tests richer window descriptors (below) chosen on development data only.
2. v3 asked retrieval to forecast the next-hour peak and compared it with persistence. Retrieval is context, not a forecaster. v4 keeps the forecast question only as an INCREMENTAL question (does the neighbour summary add information beyond the current window?) and adds the product-relevant question: when most similar past windows were followed by an Alert, is an Alert more likely next hour?

## Window descriptor candidates (declared now; the window is the last 60 bars, same causal rules as v3)
Blocks: S = 12 score summaries (mean / max / last of fused, AE, XGBoost, Transformer); ZA = max |z| of the 6 N1-normalised features; ZM = mean signed z of each feature over the window (6); ZL = signed z of each feature at the last bar (6); SH = shape of the fused score (relative position of its maximum in the window, and mean of the last 10 bars minus mean of the first 10 bars).
Candidates (all standardised per symbol with genuine DEV windows; Euclidean distance): c1 = S+ZA (the v3 descriptor); c2 = S+ZM+ZL; c3 = S+ZA+ZM; c4 = S+ZA+ZM+SH; each also with the feature blocks multiplied by 0.5 (c1w..c4w). Baselines: b1 score-only (S), b0 random past window.
Selection rule: the candidate with the highest mean precision@5 for the SAME FAMILY on the CAL-3 injected-event queries (archive = genuine past windows + labelled VAL-3 event windows, exactly as in v3). Ties go to the earlier candidate. No other criterion; the selection is made before TEST-3 / TEST-B are read. CAL-3 was already looked at once in v3 (disclosed), but it is not TEST.
Retrieval rules are unchanged: past-only (neighbour end + 60 < query start), same pair, top 5 at least 120 bars apart, reconstructed states with the frozen Watch / Alert lines, outcome = next 60 bars.
Outcome summary size k_out in {5, 10, 20} (neighbours used to summarise "what followed") is chosen on CAL genuine hourly queries by the Spearman of the combined predictor below.

## Hypotheses (TEST-3 injected events and genuine TEST-B hours; new analysis on saved scores)
- H-H1a: precision@5 "neighbour is an injected event window" >= 0.50 for the chosen candidate, queries = TEST-3 events at event end (archive = genuine past + labelled VAL-3 + labelled CAL-3 windows).
- H-H1c (replaces H-H1b): precision@5 "same family" of the chosen candidate on the same TEST-3 queries is >= 0.25 AND exceeds the score-only baseline b1 by >= 0.02 with the paired event-bootstrap 95% lower bound > 0.
- H-H2b (replaces the forecast claim of H-H2): on genuine TEST-B hourly queries a linear rank model fitted on CAL genuine hours, rank(actual next-hour peak) ~ rank(current window max fused score) + rank(mean peak of the k_out neighbours), has a Spearman with the actual peak at least 0.02 above persistence alone, day-block 95% lower bound > 0 (mean of BTC and ETH). The v3 claim H-H2 (Spearman >= 0.20 as a forecaster) stays failed and retrieval is not shown as a forecast.
- H-H5: among genuine TEST-B hourly queries, "at least 3 of the 5 neighbours were followed by an Alert within their next hour" versus the other queries: lift in the actual rate of an Alert in the next hour >= 2.0 with day-block 95% lower bound > 1 (mean of BTC and ETH).
- H-H3: no two neighbours closer than 120 bars, no neighbour or outcome bar overlapping or following the query, median query time <= 50 ms.
- H-H4: under noise sd 0.05 on the standardised descriptor, at least 60% of the top 5 remain.
- Descriptive: v3 descriptor (c1) on the same TEST-3 queries, precision by family, persistence lift for H-H5 (current window already contains an Alert), example match table.

## Known weaknesses, stated in advance
Small history (July + Aug-Oct 2024), synthetic labels for H-H1, a short-horizon outcome that is largely persistence, and eight candidate descriptors tried on one selection set. H-H1c and H-H2b may fail; they will be reported as failed and nothing will be retuned.

```

In [ ]:
import os, sys, json, time, hashlib, warnings, glob
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
PROTO_H1 = r"""# Protocol H1 v4 - historical similar-window retrieval, second version (novelty 4)
Written BEFORE any v4 result exists. Supersedes nothing: the results of H1 v3 (notebook 50: H-H1a holds, H-H1b FAILS 0.247 vs 0.25, H-H2 FAILS 0.086 vs 0.20, H-H3 holds, H-H4 holds) stay as they are and stay reported as failed. v4 is a NEW version with a changed descriptor and a changed set of claims. Nothing frozen is changed (models, 6 features, normalisation, fusion stacker, thresholds q, Layer 3 sets).

## What v3 showed, and what v4 changes
1. Feature deviations added nothing over score summaries, and "max |z|" throws away the sign and the shape of a pattern. v4 tests richer window descriptors (below) chosen on development data only.
2. v3 asked retrieval to forecast the next-hour peak and compared it with persistence. Retrieval is context, not a forecaster. v4 keeps the forecast question only as an INCREMENTAL question (does the neighbour summary add information beyond the current window?) and adds the product-relevant question: when most similar past windows were followed by an Alert, is an Alert more likely next hour?

## Window descriptor candidates (declared now; the window is the last 60 bars, same causal rules as v3)
Blocks: S = 12 score summaries (mean / max / last of fused, AE, XGBoost, Transformer); ZA = max |z| of the 6 N1-normalised features; ZM = mean signed z of each feature over the window (6); ZL = signed z of each feature at the last bar (6); SH = shape of the fused score (relative position of its maximum in the window, and mean of the last 10 bars minus mean of the first 10 bars).
Candidates (all standardised per symbol with genuine DEV windows; Euclidean distance): c1 = S+ZA (the v3 descriptor); c2 = S+ZM+ZL; c3 = S+ZA+ZM; c4 = S+ZA+ZM+SH; each also with the feature blocks multiplied by 0.5 (c1w..c4w). Baselines: b1 score-only (S), b0 random past window.
Selection rule: the candidate with the highest mean precision@5 for the SAME FAMILY on the CAL-3 injected-event queries (archive = genuine past windows + labelled VAL-3 event windows, exactly as in v3). Ties go to the earlier candidate. No other criterion; the selection is made before TEST-3 / TEST-B are read. CAL-3 was already looked at once in v3 (disclosed), but it is not TEST.
Retrieval rules are unchanged: past-only (neighbour end + 60 < query start), same pair, top 5 at least 120 bars apart, reconstructed states with the frozen Watch / Alert lines, outcome = next 60 bars.
Outcome summary size k_out in {5, 10, 20} (neighbours used to summarise "what followed") is chosen on CAL genuine hourly queries by the Spearman of the combined predictor below.

## Hypotheses (TEST-3 injected events and genuine TEST-B hours; new analysis on saved scores)
- H-H1a: precision@5 "neighbour is an injected event window" >= 0.50 for the chosen candidate, queries = TEST-3 events at event end (archive = genuine past + labelled VAL-3 + labelled CAL-3 windows).
- H-H1c (replaces H-H1b): precision@5 "same family" of the chosen candidate on the same TEST-3 queries is >= 0.25 AND exceeds the score-only baseline b1 by >= 0.02 with the paired event-bootstrap 95% lower bound > 0.
- H-H2b (replaces the forecast claim of H-H2): on genuine TEST-B hourly queries a linear rank model fitted on CAL genuine hours, rank(actual next-hour peak) ~ rank(current window max fused score) + rank(mean peak of the k_out neighbours), has a Spearman with the actual peak at least 0.02 above persistence alone, day-block 95% lower bound > 0 (mean of BTC and ETH). The v3 claim H-H2 (Spearman >= 0.20 as a forecaster) stays failed and retrieval is not shown as a forecast.
- H-H5: among genuine TEST-B hourly queries, "at least 3 of the 5 neighbours were followed by an Alert within their next hour" versus the other queries: lift in the actual rate of an Alert in the next hour >= 2.0 with day-block 95% lower bound > 1 (mean of BTC and ETH).
- H-H3: no two neighbours closer than 120 bars, no neighbour or outcome bar overlapping or following the query, median query time <= 50 ms.
- H-H4: under noise sd 0.05 on the standardised descriptor, at least 60% of the top 5 remain.
- Descriptive: v3 descriptor (c1) on the same TEST-3 queries, precision by family, persistence lift for H-H5 (current window already contains an Alert), example match table.

## Known weaknesses, stated in advance
Small history (July + Aug-Oct 2024), synthetic labels for H-H1, a short-horizon outcome that is largely persistence, and eight candidate descriptors tried on one selection set. H-H1c and H-H2b may fail; they will be reported as failed and nothing will be retuned.
"""
PHH = hashlib.sha256(PROTO_H1.encode()).hexdigest(); pp = OUT/'protocol_h1_v4.md'
if pp.exists(): assert pp.read_text() == PROTO_H1, 'protocol H1 v4 exists and differs from this notebook'
else: pp.write_text(PROTO_H1)
(OUT/'protocol_h1_v4.sha256').write_text(PHH + '  protocol_h1_v4.md\n')
for f in ['test_b_results_v1.json', 'declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'b2_lib.py', 'c03_infer.py']: assert (OUT/f).exists(), f'{f} missing'
T32 = json.loads((OUT/'test_b_results_v1.json').read_text())
for s in SYMBOLS: assert sha_file(OUT/'scores'/f'test_b_{s}.npz') == T32['test_score_files_sha16'][s], 'saved TEST scores differ from what notebook 32 recorded'
assert sha_file(OUT/'c03_infer.py') == T32['library_sha16'] and sha_file(OUT/'b2_lib.py') == T32['b2_lib_sha16'], 'library files differ from the ones that opened TEST'
sys.path.insert(0, str(OUT)); import b2_lib as B; import c03_infer as c
say('protocol H1 v4 sha256 ' + PHH[:16] + '; protocol B2 intact ' + ph[:16] + '; library ' + T32['library_sha16'] + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol H1 v4 sha256 dc8ed3be287bbb31; protocol B2 intact 443f99d5faf5929f; library 3b5d9df23521adf9


## 1 · Definitions (from notebook 50)

In [ ]:
W = 60; OUTW = 60; GAP = 120; K = 5; WARM = B.WARM; AOLEN = 31 * 1440; AOFULL = 132480; TEST_LO = AOLEN; DEV_LO, DEV_HI = 90720, 129600; DEVN = DEV_HI - DEV_LO; HIST_MIN = 1500
SERIES = ['fused', 'AE', 'XGBoost', 'Transformer']; DNAMES = [f'{s}_{st}' for s in SERIES for st in ('mean', 'max', 'last')] + [f'maxabs_z_{f}' for f in B.FEATS]; assert len(DNAMES) == 18
SPEC = {'window_bars': W, 'outcome_bars': OUTW, 'gap_bars': GAP, 'k': K, 'dims': DNAMES, 'metric': 'euclidean on per-symbol standardised descriptor', 'history_min_bars': HIST_MIN}
def raw_desc(F4, zabs, hist_min=HIST_MIN):
    """(n, 18) window descriptors for windows ending at each bar (NaN where invalid). F4 = 4 score arrays; zabs = (n, 6) |z|."""
    n = len(F4[0]); cols = []
    for x in F4:
        s = pd.Series(np.asarray(x, 'float64')); r = s.rolling(W, min_periods=W); cols += [r.mean().to_numpy(), r.max().to_numpy(), s.to_numpy()]
    zm = pd.DataFrame(np.asarray(zabs, 'float64')).rolling(W, min_periods=W).max().to_numpy(); D_ = np.column_stack(cols + [zm[:, j] for j in range(6)])
    D_[np.arange(n) < hist_min + W - 1] = np.nan; return D_
def forward_outcomes(F, thrA, logret):
    """for a window ending at e: next OUTW bars e+1..e+OUTW -> peak fused, minutes above Alert line, Alert reached, realised vol. NaN if the bars do not all exist."""
    n = len(F); pk = np.full(n, np.nan); ma = np.full(n, np.nan); vo = np.full(n, np.nan)
    if n > OUTW:
        from numpy.lib.stride_tricks import sliding_window_view as swv
        Fw = swv(F, OUTW)[1:]; Aw = swv((F > thrA).astype(float), OUTW)[1:]; Lw = swv(logret, OUTW)[1:]; m = len(Fw)
        pk[:m] = Fw.max(axis=1); ma[:m] = Aw.sum(axis=1); vo[:m] = Lw.std(axis=1)
    return pk, ma, vo
def select(dist, times, k=K, gap=GAP):
    """indices of up to k nearest candidates, nearest first, every pair >= gap bars apart"""
    order = np.argsort(dist, kind='stable'); picked = []
    for i in order:
        if all(abs(times[i] - times[j]) >= gap for j in picked): picked.append(i)
        if len(picked) == k: break
    return picked
def retrieve(A, At, q, q_end, k=K, gap=GAP):
    """A: standardised archive (n, d) sorted by end time At; q: query descriptor; past-only: archive end + OUTW < query start (q_end - W + 1)"""
    n = int(np.searchsorted(At, q_end - W + 1 - OUTW, side='left'))      # At < q_start - OUTW
    if n == 0: return np.array([], int), np.array([])
    d = np.sqrt(((A[:n] - q) ** 2).sum(axis=1)); m = min(n, 600); part = np.argpartition(d, m - 1)[:m] if n > m else np.arange(n)
    sel = select(d[part], At[part], k, gap)
    if len(sel) < k and n > m: sel = select(d, At[:n], k, gap); return np.array(sel, int), d[sel]
    ii = part[sel]; return np.array(ii, int), d[ii]

## 2 · Genuine timeline (from notebook 50)

In [ ]:
DF = json.load(open(OUT/'declared_final_set_v1.json')); FUSD = json.load(open(OUT/'declared_fusion_v0.json')); DW = json.load(open(OUT/'declared_watch_v1.json'))
MODELS = c.MODEL_NAMES; FNORM = {m: DF['models'][m]['normalisation'] for m in MODELS}; FILES = {'AE-ctx fmax': 'ae_ctx_agg', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}; TAGM = dict(zip(MODELS, ['AE', 'XGB', 'TR']))
QA = FUSD['q_T1_by_budget']['2']; QW = DW['watch']['q']['4']
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:WARM] = np.nan; return a
def find1(pat):
    cc = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert cc, pat; return sorted(cc)[0]
def fix(d): d = d.copy(); d.loc[d.index[:15], 'volume_acceleration'] = 0.0; return d
def mcol(zz, k): x = zz[k]; return x[:, 0] if x.ndim == 2 else x
G = {}      # per symbol: genuine timeline (DEV then Aug-Oct) and injected series for the H-H1 evaluation
for s in SYMBOLS:
    a = fix(pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet'))); a = a.set_index('datetime') if 'datetime' in a.columns else a
    b = fix(pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet')); assert len(b) == AOFULL, len(b)
    N0 = B.fit_n0(a[B.FEATS].reset_index(drop=True).iloc[15:90720])
    zL1 = np.abs(B.normalise(a[B.FEATS].reset_index(drop=True), 'N1', N0)); zAO = np.abs(B.normalise(b[B.FEATS].reset_index(drop=True), 'N1', N0))
    zf = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); tz = np.load(OUT/'scores'/f'test_b_{s}.npz'); assert 'FUS__GEN' in tz.files
    dev = [nanw(zf['OOF_GEN_L1'])[DEV_LO:DEV_HI]]; ao = [nanw(tz['FUS__GEN'])]
    for m in MODELS:
        zz = np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz'); dev.append(nanw(mcol(zz, 'GEN_L1'))[DEV_LO:DEV_HI]); ao.append(nanw(tz[TAGM[m] + '__GEN']))
    # parity: the August part of the saved TEST file equals the fusion August scores of notebook 35
    dmax = np.nanmax(np.abs(ao[0][:AOLEN] - nanw(zf['GEN_AO'])[:AOLEN])); say(f'{s}: parity of August fused scores (TEST file vs notebook 35): max difference {dmax:.2e}')
    ts = np.concatenate([a.index.to_numpy()[DEV_LO:DEV_HI], b.index.to_numpy()]).astype('datetime64[ns]')
    Fg = np.concatenate([dev[0], ao[0]]); thrA = c.rolling_threshold(Fg, QA[s]); thrW = c.rolling_threshold(Fg, QW[s])
    seg = np.concatenate([np.zeros(DEVN, int), np.ones(AOFULL, int)]); lr = np.concatenate([a['log_returns'].to_numpy()[DEV_LO:DEV_HI], b['log_returns'].to_numpy()])
    RD = np.vstack([raw_desc([x for x in [dev[0]] + dev[1:]], zL1[DEV_LO:DEV_HI], hist_min=0), raw_desc(ao[0:1] + ao[1:], zAO, hist_min=HIST_MIN)])
    pk, ma, vo = forward_outcomes(Fg[:DEVN], thrA[:DEVN], lr[:DEVN]); pk2, ma2, vo2 = forward_outcomes(Fg[DEVN:], thrA[DEVN:], lr[DEVN:]); PK, MA, VO = np.concatenate([pk, pk2]), np.concatenate([ma, ma2]), np.concatenate([vo, vo2])
    st = np.where(Fg > thrA, 2, np.where(Fg > thrW, 1, 0)); stp = pd.Series(st).rolling(W, min_periods=1).max().to_numpy().astype(int)
    G[s] = dict(ts=ts, F=Fg, thrA=thrA, thrW=thrW, seg=seg, lr=lr, RD=RD, PK=PK, MA=MA, VO=VO, st=st, stp=stp, N0=N0, parts=(dev, ao), zL1=zL1, zAO=zAO, a=a, b=b)
    say(f'{s}: genuine timeline {len(Fg)} bars (DEV {DEVN} + Aug-Oct {AOFULL}); valid windows with outcome {int((np.isfinite(RD).all(axis=1) & np.isfinite(PK)).sum())}')

BTCUSDT: parity of August fused scores (TEST file vs notebook 35): max difference 9.61e-03
BTCUSDT: genuine timeline 171360 bars (DEV 38880 + Aug-Oct 132480); valid windows with outcome 169622
ETHUSDT: parity of August fused scores (TEST file vs notebook 35): max difference 7.44e-03
ETHUSDT: genuine timeline 171360 bars (DEV 38880 + Aug-Oct 132480); valid windows with outcome 169622


## 3 · Extended window descriptors

In [ ]:
# ---------------- extended descriptors (blocks S, ZA, ZM, ZL, SH) on the genuine timeline
from numpy.lib.stride_tricks import sliding_window_view as swv
BL = {'S': np.arange(0, 12), 'ZA': np.arange(12, 18), 'ZM': np.arange(18, 24), 'ZL': np.arange(24, 30), 'SH': np.arange(30, 32)}
def ext_desc(F4, zs, hist_min=HIST_MIN):
    """(n, 32): 12 score summaries | 6 max|z| | 6 mean signed z | 6 last signed z | 2 shape numbers; NaN where invalid"""
    zs = np.asarray(zs, 'float64'); n = len(F4[0]); base = raw_desc(F4, np.abs(zs), hist_min); ok = np.isfinite(base).all(axis=1)
    zm = pd.DataFrame(zs).rolling(W, min_periods=W).mean().to_numpy(); f = np.asarray(F4[0], 'float64'); pos = np.full(n, np.nan)
    if n >= W: pos[W - 1:] = np.argmax(swv(np.nan_to_num(f, nan=-1e9), W), axis=1) / (W - 1)
    m10 = pd.Series(f).rolling(10, min_periods=10).mean().to_numpy(); slope = m10 - pd.Series(m10).shift(W - 10).to_numpy()
    D = np.column_stack([base, zm, zs, pos, slope]); D[~ok] = np.nan; return D
CANDS = {}
for nm, blocks in [('c1', ['S', 'ZA']), ('c2', ['S', 'ZM', 'ZL']), ('c3', ['S', 'ZA', 'ZM']), ('c4', ['S', 'ZA', 'ZM', 'SH'])]:
    CANDS[nm] = (blocks, 1.0); CANDS[nm + 'w'] = (blocks, 0.5)
CANDS['b1 score-only'] = (['S'], 1.0)
def cspec(c):
    blocks, w = CANDS[c]; cols = np.concatenate([BL[b] for b in blocks]); wt = np.concatenate([np.full(len(BL[b]), 1.0 if b == 'S' else w) for b in blocks]); return cols, wt
EXS = {}; STDE = {}; ARC = {}
for s in SYMBOLS:
    g = G[s]; dev, ao = g['parts']; zL = np.asarray(B.normalise(g['a'][B.FEATS].reset_index(drop=True), 'N1', g['N0']), 'float64'); zA = np.asarray(B.normalise(g['b'][B.FEATS].reset_index(drop=True), 'N1', g['N0']), 'float64')
    EX = np.vstack([ext_desc(dev, zL[DEV_LO:DEV_HI], 0), ext_desc(ao, zA, HIST_MIN)])
    valid = np.isfinite(EX).all(axis=1) & np.isfinite(g['PK']) & np.isfinite(g['MA']) & np.isfinite(g['VO']); devm = valid & (np.arange(len(valid)) < DEVN)
    mu = EX[devm].mean(axis=0); sd = EX[devm].std(axis=0) + 1e-9; STDE[s] = (mu, sd); Z = (EX - mu) / sd; idx = np.flatnonzero(valid); EXS[s] = Z
    ARC[s] = dict(Z=Z[idx], t=idx.astype(np.int64), PK=g['PK'][idx], MA=g['MA'][idx], lab=np.zeros(len(idx), int))
    say(f'{s}: extended descriptors (32 numbers); standardiser from {int(devm.sum())} genuine DEV windows; genuine archive {len(idx)} windows')

BTCUSDT: extended descriptors (32 numbers); standardiser from 38761 genuine DEV windows; genuine archive 169622 windows
ETHUSDT: extended descriptors (32 numbers); standardiser from 38761 genuine DEV windows; genuine archive 169622 windows


## 4 · Unit tests

In [ ]:
rg = np.random.default_rng(0); n = 3000; F4 = [rg.normal(size=n).cumsum() * 0.01 + i for i in range(4)]; zs_ = rg.normal(size=(n, 6)); Dd = ext_desc(F4, zs_, 0); e = 2000
assert np.allclose(Dd[e, 18:24], zs_[e - W + 1:e + 1].mean(axis=0)) and np.allclose(Dd[e, 24:30], zs_[e]) and abs(Dd[e, 30] - np.argmax(F4[0][e - W + 1:e + 1]) / (W - 1)) < 1e-12 and abs(Dd[e, 31] - (F4[0][e - 9:e + 1].mean() - F4[0][e - W + 1:e - W + 11].mean())) < 1e-12
assert np.allclose(Dd[e, :18], raw_desc(F4, np.abs(zs_), 0)[e]); say('PASS T01 extended descriptor (mean / last signed z, shape numbers) equals a hand calculation and contains the v3 descriptor')
F4b = [np.where(np.arange(n) > e, x * 5 + 3, x) for x in F4]; zb = np.where(np.arange(n)[:, None] > e, zs_ * 4, zs_); Db = ext_desc(F4b, zb, 0); assert np.allclose(Dd[:e + 1], Db[:e + 1], equal_nan=True); say('PASS T02 the descriptor at bar t does not depend on later bars')
cols, wt = cspec('c1'); assert list(cols) == list(range(18)) and (wt == 1).all(); cols, wt = cspec('c4w'); assert len(cols) == 26 and wt[:12].min() == 1 and wt[12:].max() == 0.5; assert cspec('b1 score-only')[0].tolist() == list(range(12)); say('PASS T03 candidate c1 is exactly the v3 descriptor; weights apply to feature blocks only')
for s in SYMBOLS:
    t_ = ARC[s]['t']; assert not ((t_ >= DEVN - OUTW) & (t_ < DEVN + HIST_MIN + W - 1)).any() and np.isfinite(ARC[s]['Z']).all()
    dm = np.isfinite(G[s]['RD'][:DEVN]).all(axis=1) & np.isfinite(G[s]['PK'][:DEVN]); assert STDE[s][0].shape == (32,)
say('PASS T04 no archive window spans the seam or lacks history; standardiser from genuine DEV windows only')
def _ref(ZA_, At, q, q_end):
    cand = np.flatnonzero(At + OUTW < q_end - W + 1); dist = np.sqrt(((ZA_[cand] - q) ** 2).sum(axis=1)); ref = []
    for j in np.argsort(dist, kind='stable'):
        if all(abs(At[cand[j]] - At[r]) >= GAP for r in ref): ref.append(cand[j])
        if len(ref) == 20: break
    return ref
At = np.unique(np.sort(rg.integers(0, 50000, 4000))); A_ = rg.normal(size=(len(At), 32)); q = rg.normal(size=32); ii, dd = retrieve(A_, At, q, 33333, k=20); assert list(ii) == [int(x) for x in _ref(A_, At, q, 33333)] and list(retrieve(A_, At, q, 33333, k=5)[0]) == list(ii[:5])
say('PASS T05 top-20 retrieval equals brute force and its first 5 equal the top-5 retrieval')

PASS T01 extended descriptor (mean / last signed z, shape numbers) equals a hand calculation and contains the v3 descriptor
PASS T02 the descriptor at bar t does not depend on later bars
PASS T03 candidate c1 is exactly the v3 descriptor; weights apply to feature blocks only
PASS T04 no archive window spans the seam or lacks history; standardiser from genuine DEV windows only
PASS T05 top-20 retrieval equals brute force and its first 5 equal the top-5 retrieval


## 5 · Labelled injected-event windows (VAL-3, CAL-3, TEST-3)

In [ ]:
# ---------------- labelled injected-event windows (VAL-3, CAL-3, TEST-3), same inclusion rule as v3
def win_from(f, ev, sc, lo, hi, off, hist, N0, mu, sd):
    zs = np.asarray(B.normalise(f[B.FEATS].reset_index(drop=True), 'N1', N0), 'float64')[lo:hi]; Rv = ext_desc([x[lo:hi] for x in sc], zs, hist); lab = f.label.to_numpy().astype(int)[lo:hi]; n = hi - lo; out = []
    for e_ in ev.itertuples():
        e = e_.t0 + e_.dur - 1 - lo
        if e < 0 or e >= n or not np.isfinite(Rv[e]).all(): continue
        wl = set(np.unique(lab[max(e - W + 1, 0):e + 1])) - {0}
        if wl and wl != {int(e_.family)}: continue
        out.append((off + e, (Rv[e] - mu) / sd, int(e_.family)))
    return out
LABW = {}
def stack(lst): return (np.array([x[0] for x in lst], np.int64), np.array([x[1] for x in lst]).reshape(len(lst), 32), np.array([x[2] for x in lst], int))
for s in SYMBOLS:
    g = G[s]; mu, sd = STDE[s]; zf = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); tz = np.load(OUT/'scores'/f'test_b_{s}.npz'); mf = {m: np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz') for m in MODELS}
    vp = pd.read_parquet(L3/f'l3v3_{s}_VAL.parquet'); ve = pd.read_csv(L3/f'l3v3_{s}_VAL_events.csv'); V = []
    for r in sorted(vp.replica.unique()):
        f = vp[vp.replica == r].reset_index(drop=True); sc = [nanw(zf[f'OOF_VAL_r{r}'])] + [nanw(mcol(mf[m], f'VAL_r{r}')) for m in MODELS]; V += win_from(f, ve[ve.replica == r], sc, DEV_LO, DEV_HI, 0, 0, g['N0'], mu, sd)
    cp = pd.read_parquet(L3/f'l3v3_{s}_CAL.parquet'); cp = cp[cp.replica == 0].reset_index(drop=True).iloc[:AOLEN]; ce = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); ce = ce[ce.replica == 0]
    Cw = win_from(cp, ce, [nanw(zf['CAL_r0'][:AOLEN])] + [nanw(mcol(mf[m], 'CAL_r0')[:AOLEN]) for m in MODELS], 0, AOLEN, DEVN, HIST_MIN, g['N0'], mu, sd)
    tp = pd.read_parquet(L3/f'l3v3_{s}_TEST.parquet'); te = pd.read_csv(L3/f'l3v3_{s}_TEST_events.csv'); Tw = []
    for r in sorted(tp.replica.unique()):
        f = tp[tp.replica == r].reset_index(drop=True); sc = [nanw(tz[f'FUS__r{r}'])] + [nanw(tz[f'{TAGM[m]}__r{r}']) for m in MODELS]; Tw += win_from(f, te[te.replica == r], sc, 0, AOFULL, DEVN, HIST_MIN, g['N0'], mu, sd)
    LABW[s] = dict(VAL=stack(V), CAL=stack(Cw), TEST=stack(Tw)); say(f'{s}: labelled event windows VAL {len(V)}, CAL {len(Cw)}, TEST-3 {len(Tw)}')
def archive(s, parts):
    A = ARC[s]; t = [A['t']] + [LABW[s][p][0] for p in parts]; Z = [A['Z']] + [LABW[s][p][1] for p in parts]; L = [A['lab']] + [LABW[s][p][2] for p in parts]; t = np.concatenate(t); o = np.argsort(t, kind='stable'); return np.vstack(Z)[o], t[o], np.concatenate(L)[o]

BTCUSDT: labelled event windows VAL 295, CAL 198, TEST-3 298
ETHUSDT: labelled event windows VAL 296, CAL 198, TEST-3 298


## 6 · Candidate selection on CAL-3 (development data)

In [ ]:
# ---------------- candidate selection on CAL-3 queries (archive = genuine past + labelled VAL-3), TEST untouched
def hits(s, cand, qset, arch_parts, rng=None):
    Z, t, lab = archive(s, arch_parts); cols, wt = cspec(cand) if cand != 'b0 random past' else (np.arange(32), np.ones(32)); Zw = Z[:, cols] * wt; qt, qz, qf = LABW[s][qset]; out = []
    for e, qd, fam in zip(qt, qz, qf):
        if cand == 'b0 random past':
            n = int(np.searchsorted(t, e - W + 1 - OUTW, side='left')); pool = rng.permutation(n); ii = pool[select(np.zeros(n), t[:n][pool], K, GAP)]
        else: ii, _ = retrieve(Zw, t, qd[cols] * wt, e, k=K)
        if len(ii) == 0: out.append((np.nan, np.nan)); continue
        l_ = lab[ii]; out.append((float((l_ > 0).mean()), float((l_ == fam).mean())))
    return np.array(out)
SEL = []
for cand in [c_ for c_ in CANDS if c_ != 'b1 score-only'] + ['b1 score-only']:
    h = [hits(s, cand, 'CAL', ['VAL']) for s in SYMBOLS]; SEL.append({'candidate': cand, 'blocks': '+'.join(CANDS[cand][0]), 'feature weight': CANDS[cand][1], 'CAL-3 P@5 any event': float(np.nanmean(np.concatenate(h)[:, 0])), 'CAL-3 P@5 same family': float(np.nanmean(np.concatenate(h)[:, 1]))})
SELT = pd.DataFrame(SEL); SELT.to_csv(OUT/'h1v4_selection_table.csv', index=False); main = SELT[SELT.candidate != 'b1 score-only']; CHOSEN = main.sort_values('CAL-3 P@5 same family', ascending=False, kind='stable').iloc[0].candidate
say('--- candidate selection on CAL-3 injected-event queries (development data; TEST not used) ---'); say(SELT.round(3).to_string(index=False)); say(f'chosen candidate (highest same-family P@5): {CHOSEN}')

--- candidate selection on CAL-3 injected-event queries (development data; TEST not used) ---
    candidate     blocks  feature weight  CAL-3 P@5 any event  CAL-3 P@5 same family
           c1       S+ZA             1.0                0.584                  0.249
          c1w       S+ZA             0.5                0.589                  0.251
           c2    S+ZM+ZL             1.0                0.580                  0.290
          c2w    S+ZM+ZL             0.5                0.587                  0.263
           c3    S+ZA+ZM             1.0                0.594                  0.274
          c3w    S+ZA+ZM             0.5                0.594                  0.260
           c4 S+ZA+ZM+SH             1.0                0.607                  0.277
          c4w S+ZA+ZM+SH             0.5                0.604                  0.264
b1 score-only          S             1.0                0.590                  0.244
chosen candidate (highest same-family P@5): c2


## 7 · Versioned index

In [ ]:
cols, wt = cspec(CHOSEN); SPEC4 = {'window_bars': W, 'outcome_bars': OUTW, 'gap_bars': GAP, 'k': K, 'candidate': CHOSEN, 'blocks': CANDS[CHOSEN][0], 'feature_weight': CANDS[CHOSEN][1], 'dims': int(len(cols)), 'metric': 'euclidean on per-symbol standardised, block-weighted descriptor'}
def idx_version(s):
    h = hashlib.sha256(); h.update(json.dumps(SPEC4, sort_keys=True).encode()); h.update(STDE[s][0].tobytes()); h.update(STDE[s][1].tobytes()); h.update(ARC[s]['t'].tobytes()); return h.hexdigest()
VERS = {s: idx_version(s) for s in SYMBOLS}
for s in SYMBOLS:
    A = ARC[s]; np.savez_compressed(OUT/f'history_index_v2_{s}.npz', Z=(A['Z'][:, cols] * wt).astype('float32'), t=A['t'], PK=A['PK'], MA=A['MA'], ts=G[s]['ts'][A['t']].astype('datetime64[ns]').astype('int64'), mean=STDE[s][0], std=STDE[s][1], cols=cols, weights=wt)
(OUT/'history_index_v2.json').write_text(json.dumps({'spec': SPEC4, 'version_sha256': VERS, 'protocol_h1_v4_sha256': PHH, 'note': 'index files are data: kept on Drive, never committed'}, indent=1))
assert VERS == {s: idx_version(s) for s in SYMBOLS}; say('PASS T06 index version hash is deterministic; versions ' + str({s: VERS[s][:16] for s in SYMBOLS}))

PASS T06 index version hash is deterministic; versions {'BTCUSDT': 'a8f3c79ee27d9fab', 'ETHUSDT': 'f86ed53c57738bcf'}


## 8 · H-H1 on held-out TEST-3 events

In [ ]:
# ---------------- H-H1a / H-H1c on TEST-3 injected events (held-out confirmation; archive = genuine past + labelled VAL-3 + CAL-3)
def pair_boot(x, y, n=NBOOT, seed=0):
    rg = np.random.default_rng(seed); d = x - y; m = len(d); bs = [d[rg.integers(0, m, m)].mean() for _ in range(n)]; return float(d.mean()), np.percentile(bs, [2.5, 97.5])
VARS = [CHOSEN, 'c1', 'b1 score-only', 'b0 random past']; HR = {}; rows = []
for v in VARS:
    h = [hits(s, v, 'TEST', ['VAL', 'CAL'], np.random.default_rng(1)) for s in SYMBOLS]; HR[v] = np.concatenate(h)
    for s, hh in zip(SYMBOLS, h): rows.append({'variant': v, 'sym': s, 'queries': len(hh), 'P@5 any event window': float(np.nanmean(hh[:, 0])), 'P@5 same family': float(np.nanmean(hh[:, 1]))})
T3 = pd.DataFrame(rows); T3.to_csv(OUT/'h1v4_test3_table.csv', index=False); say(''); say('--- H-H1 on TEST-3 injected events (held-out) ---'); say(T3.round(3).to_string(index=False))
ok = np.isfinite(HR[CHOSEN][:, 1]) & np.isfinite(HR['b1 score-only'][:, 1]); H1a = float(np.nanmean(HR[CHOSEN][:, 0])); H1c_v = float(np.nanmean(HR[CHOSEN][:, 1]))
dm, dci = pair_boot(HR[CHOSEN][ok, 1], HR['b1 score-only'][ok, 1]); dm3, dci3 = pair_boot(HR[CHOSEN][ok, 1], HR['c1'][ok, 1]) if CHOSEN != 'c1' else (0.0, np.array([0.0, 0.0]))
HH1a = bool(H1a >= 0.50); HH1c = bool(H1c_v >= 0.25 and dm >= 0.02 and dci[0] > 0)
fam = pd.DataFrame({'fam': np.concatenate([LABW[s]['TEST'][2] for s in SYMBOLS]), 'any': HR[CHOSEN][:, 0], 'same': HR[CHOSEN][:, 1]}).groupby('fam').mean(); fam.index = [B.FAMS[int(i)] for i in fam.index]; say(''); say('chosen candidate by family (TEST-3, both symbols):'); say(fam.round(3).to_string())
say(f'chosen {CHOSEN}: same-family P@5 {H1c_v:.3f}; minus score-only {dm:+.3f} (paired 95% {dci[0]:+.3f}..{dci[1]:+.3f}); minus v3 descriptor c1 {dm3:+.3f} ({dci3[0]:+.3f}..{dci3[1]:+.3f})')


--- H-H1 on TEST-3 injected events (held-out) ---
       variant     sym  queries  P@5 any event window  P@5 same family
            c2 BTCUSDT      298                 0.553            0.313
            c2 ETHUSDT      298                 0.532            0.283
            c1 BTCUSDT      298                 0.565            0.289
            c1 ETHUSDT      298                 0.550            0.276
 b1 score-only BTCUSDT      298                 0.558            0.282
 b1 score-only ETHUSDT      298                 0.544            0.260
b0 random past BTCUSDT      298                 0.006            0.001
b0 random past ETHUSDT      298                 0.005            0.002

chosen candidate by family (TEST-3, both symbols):
                             any   same
pump_and_dump_like         0.521  0.189
fake_breakout_like         0.462  0.200
abnormal_volume_burst      0.035  0.011
trade_flow_imbalance       0.645  0.382
wash_like_aggregate_proxy  0.978  0.762
liquidity_stress_p

## 9 · Genuine hourly queries: H-H2b, H-H5, H-H3, H-H4

In [ ]:
# ---------------- genuine hourly queries: CAL (fit / choose k_out) and TEST-B (evaluate); archive = genuine windows, past-only
from scipy.stats import rankdata
KOUT = [5, 10, 20]; cols, wt = cspec(CHOSEN)
def run_queries(s, lo, hi, record=False):
    g = G[s]; A = ARC[s]; Zw = A['Z'][:, cols] * wt; qe = np.arange(lo, hi); qe = qe[(qe - DEVN) % 60 == 0]; qe = qe[np.isfinite(EXS[s][qe]).all(axis=1) & np.isfinite(g['PK'][qe])]
    out = dict(q=qe, act=g['PK'][qe], actA=(g['MA'][qe] > 0).astype(float), pers=g['RD'][qe, 1], persA=(g['F'][qe] > -np.inf) * 0.0, nb={k: np.full(len(qe), np.nan) for k in KOUT}, a5=np.full(len(qe), np.nan), lat=[], dup=0, leak=0, ovl=[])
    wA = pd.Series((g['F'] > g['thrA']).astype(float)).rolling(W, min_periods=1).max().to_numpy(); out['persA'] = wA[qe]
    for j, e in enumerate(qe):
        q = (EXS[s][e][cols]) * wt; t0_ = time.perf_counter(); ii, dd = retrieve(Zw, A['t'], q, e, k=max(KOUT)); lat = time.perf_counter() - t0_
        if len(ii) == 0: continue
        for k in KOUT: out['nb'][k][j] = A['PK'][ii[:k]].mean()
        out['a5'][j] = (A['MA'][ii[:5]] > 0).sum() if len(ii) >= 5 else np.nan
        if record:
            T_ = A['t'][ii[:5]]; out['lat'].append(lat); out['leak'] += int((T_ + OUTW >= e - W + 1).sum()); out['dup'] += int(sum(abs(T_[a_] - T_[b_]) < GAP for a_ in range(len(T_)) for b_ in range(a_)))
            if j % 25 == 0:
                i2, _ = retrieve(Zw, A['t'], q + np.random.default_rng(j).normal(0, 0.05, len(q)), e, k=5); out['ovl'].append(len(set(ii[:5].tolist()) & set(i2.tolist())) / 5)
    return out
CALQ = {s: run_queries(s, DEVN + 1440, DEVN + AOLEN - OUTW - 1) for s in SYMBOLS}
TSTQ = {s: run_queries(s, DEVN + TEST_LO + 1440, DEVN + AOFULL - OUTW - 1, record=True) for s in SYMBOLS}
def prank(x): return (rankdata(x) - 0.5) / len(x)
def fit_rank(Q, k):
    ok = np.isfinite(Q['nb'][k]) & np.isfinite(Q['act']); X = np.column_stack([np.ones(ok.sum()), prank(Q['pers'][ok]), prank(Q['nb'][k][ok])]); co = np.linalg.lstsq(X, prank(Q['act'][ok]), rcond=None)[0]; return co
def combo(Q, k, co): ok = np.isfinite(Q['nb'][k]) & np.isfinite(Q['act']); return ok, co[0] + co[1] * prank(Q['pers'][ok]) + co[2] * prank(Q['nb'][k][ok])
KS = {}
for k in KOUT:
    v = []
    for s in SYMBOLS:
        co = fit_rank(CALQ[s], k); ok, p_ = combo(CALQ[s], k, co); v.append(spearmanr(p_, CALQ[s]['act'][ok])[0])
    KS[k] = float(np.mean(v))
KBEST = max(KS, key=lambda k: KS[k]); say(''); say('k_out chosen on CAL genuine hourly queries (Spearman of the combined predictor): ' + str({k: round(v, 3) for k, v in KS.items()}) + f' -> k_out = {KBEST}')
COEF = {s: fit_rank(CALQ[s], KBEST) for s in SYMBOLS}
def sp_diff_for(sel):
    d = []
    for s in SYMBOLS:
        Q = TSTQ[s]; ok, p_ = combo(Q, KBEST, COEF[s]); h = sel[s][ok]; d.append(spearmanr(p_[h], Q['act'][ok][h])[0] - spearmanr(Q['pers'][ok][h], Q['act'][ok][h])[0])
    return float(np.mean(d))
dayof = {s: ((TSTQ[s]['q'] - DEVN - TEST_LO) // 1440) for s in SYMBOLS}
def day_boot(fn, n=NBOOT, seed=0):
    ud = {s: np.unique(dayof[s]) for s in SYMBOLS}; rg = np.random.default_rng(seed); out = []
    for _ in range(n):
        sel = {}
        for s in SYMBOLS:
            ok, _p = combo(TSTQ[s], KBEST, COEF[s]); pick = rg.choice(ud[s], len(ud[s])); idx = np.concatenate([np.flatnonzero(dayof[s] == d) for d in pick]); m = np.zeros(len(dayof[s]), bool); sel[s] = (idx, ok)
        out.append(fn(sel))
    return np.nanpercentile(out, [2.5, 97.5])
def gain_fn(sel):
    d = []
    for s in SYMBOLS:
        idx, ok = sel[s]; Q = TSTQ[s]; okk, p_ = combo(Q, KBEST, COEF[s]); full = np.full(len(Q['q']), np.nan); full[okk] = p_; ii = idx[np.isfinite(full[idx])]
        d.append(spearmanr(full[ii], Q['act'][ii])[0] - spearmanr(Q['pers'][ii], Q['act'][ii])[0])
    return float(np.mean(d))
GAIN = float(np.mean([spearmanr(combo(TSTQ[s], KBEST, COEF[s])[1], TSTQ[s]['act'][combo(TSTQ[s], KBEST, COEF[s])[0]])[0] - spearmanr(TSTQ[s]['pers'][combo(TSTQ[s], KBEST, COEF[s])[0]], TSTQ[s]['act'][combo(TSTQ[s], KBEST, COEF[s])[0]])[0] for s in SYMBOLS]))
cg = day_boot(gain_fn); HH2b = bool(GAIN >= 0.02 and cg[0] > 0)
SPr = {s: (spearmanr(TSTQ[s]['pers'], TSTQ[s]['act'])[0], spearmanr(TSTQ[s]['nb'][KBEST][np.isfinite(TSTQ[s]['nb'][KBEST])], TSTQ[s]['act'][np.isfinite(TSTQ[s]['nb'][KBEST])])[0]) for s in SYMBOLS}
say('--- H-H2b · does the neighbour summary add information beyond persistence? (genuine TEST-B hourly windows) ---'); say(f'Spearman with the actual next-hour peak: persistence {np.mean([v[0] for v in SPr.values()]):.3f}, neighbours alone {np.mean([v[1] for v in SPr.values()]):.3f}; combined minus persistence {GAIN:+.3f} (day-block 95% {cg[0]:+.3f}..{cg[1]:+.3f})')
# H-H5: >= 3 of 5 neighbours were followed by an Alert -> actual Alert in the next hour
def lift5(sel=None):
    out = []
    for s in SYMBOLS:
        Q = TSTQ[s]; ok = np.isfinite(Q['a5']); idx = np.arange(len(Q['q'])) if sel is None else sel[s][0]; idx = idx[ok[idx]]; grp = Q['a5'][idx] >= 3; y = Q['actA'][idx]
        out.append(y[grp].mean() / max(y[~grp].mean(), 1e-12) if grp.any() and (~grp).any() and y[~grp].mean() > 0 else np.nan)
    return float(np.nanmean(out)) if np.isfinite(out).any() else np.nan
def liftP(sel=None):
    out = []
    for s in SYMBOLS:
        Q = TSTQ[s]; idx = np.arange(len(Q['q'])) if sel is None else sel[s][0]; grp = Q['persA'][idx] > 0; y = Q['actA'][idx]
        out.append(y[grp].mean() / max(y[~grp].mean(), 1e-12) if grp.any() and (~grp).any() and y[~grp].mean() > 0 else np.nan)
    return float(np.nanmean(out)) if np.isfinite(out).any() else np.nan
L5 = lift5(); c5 = day_boot(lift5); LPp = liftP(); HH5 = bool(np.isfinite(L5) and L5 >= 2.0 and c5[0] > 1)
grp_share = float(np.mean(np.concatenate([(TSTQ[s]['a5'][np.isfinite(TSTQ[s]['a5'])] >= 3) for s in SYMBOLS])))
say(f'H-H5 queries whose neighbours were mostly followed by an Alert: {grp_share:.3f} of hours; actual Alert-next-hour lift {L5:.2f} (day-block {c5[0]:.2f}..{c5[1]:.2f}); (descriptive) persistence rule "current window already contains an Alert" lift {LPp:.2f}')
LATMS = float(np.median(np.concatenate([TSTQ[s]['lat'] for s in SYMBOLS])) * 1000); DUP = sum(TSTQ[s]['dup'] for s in SYMBOLS); LEAK = sum(TSTQ[s]['leak'] for s in SYMBOLS); OV = float(np.mean(np.concatenate([TSTQ[s]['ovl'] for s in SYMBOLS])))
HH3 = bool(DUP == 0 and LEAK == 0 and LATMS <= 50); HH4 = bool(OV >= 0.60)
say(f'H-H3 duplicates {DUP}, leakage {LEAK}, median query {LATMS:.1f} ms; H-H4 top-5 overlap under noise {OV:.3f}')


k_out chosen on CAL genuine hourly queries (Spearman of the combined predictor): {5: 0.257, 10: 0.277, 20: 0.282} -> k_out = 20
--- H-H2b · does the neighbour summary add information beyond persistence? (genuine TEST-B hourly windows) ---
Spearman with the actual next-hour peak: persistence 0.087, neighbours alone 0.148; combined minus persistence +0.052 (day-block 95% +0.030..+0.072)
H-H5 queries whose neighbours were mostly followed by an Alert: 0.005 of hours; actual Alert-next-hour lift 1.97 (day-block 0.00..6.19); (descriptive) persistence rule "current window already contains an Alert" lift 1.85
H-H3 duplicates 0, leakage 0, median query 13.7 ms; H-H4 top-5 overlap under noise 0.853


## 10 · Example match table

In [ ]:
s = SYMBOLS[0]; Q = TSTQ[s]; A = ARC[s]; Zw = A['Z'][:, cols] * wt; g = G[s]; names = {0: 'NORMAL', 1: 'WATCH', 2: 'ALERT'}
cand = np.flatnonzero(np.isfinite(Q['act'])); j = int(cand[np.argmax(Q['act'][cand])]); e = int(Q['q'][j]); ii, dd = retrieve(Zw, A['t'], EXS[s][e][cols] * wt, e, k=5); n_ = int(np.searchsorted(A['t'], e - W + 1 - OUTW, side='left')); dall = np.sqrt(((Zw[:n_] - EXS[s][e][cols] * wt) ** 2).sum(axis=1))
rows = [{'match': i + 1, 'window ended (UTC)': str(g['ts'][A['t'][k_]])[:16], 'closer than % of past windows': round(100 * float((dall > d).mean()), 1), 'state (reconstructed)': names[int(g['st'][A['t'][k_]])], 'peak fused next hour': round(float(A['PK'][k_]), 2), 'minutes above Alert next hour': int(A['MA'][k_])} for i, (k_, d) in enumerate(zip(ii, dd))]
say(''); say(f'--- example: {s} hour ended {str(g["ts"][e])[:16]} UTC (actual next-hour peak {Q["act"][j]:.2f}); 5 most similar PAST windows ---'); say(pd.DataFrame(rows).to_string(index=False)); say('Similar past patterns do not guarantee the same future. Not a price forecast.')


--- example: BTCUSDT hour ended 2024-09-27T07:00 UTC (actual next-hour peak 5.82); 5 most similar PAST windows ---
 match window ended (UTC)  closer than % of past windows state (reconstructed)  peak fused next hour  minutes above Alert next hour
     1   2024-07-25T13:50                          100.0                NORMAL                  1.45                              0
     2   2024-09-09T02:35                          100.0                NORMAL                  1.79                              0
     3   2024-08-27T11:36                          100.0                NORMAL                  1.31                              0
     4   2024-08-15T23:00                          100.0                NORMAL                  0.69                              0
     5   2024-07-19T10:02                          100.0                NORMAL                 -0.38                              0
Similar past patterns do not guarantee the same future. Not a price forecast.


## 11 · Results (protocol H1 v4) and save

In [ ]:
say(''); say('=== pre-registered hypotheses (protocol H1 v4; new analysis on saved TEST-3 / TEST-B scores; v3 results are unchanged and stay as reported) ===')
say(f'H-H1a precision@5 "event window" >= 0.50 (TEST-3, chosen {CHOSEN}): {H1a:.3f} -> {"HOLDS" if HH1a else "FAILS"}')
say(f'H-H1c same-family P@5 >= 0.25 and >= score-only + 0.02 (paired lower bound > 0): {H1c_v:.3f}; vs score-only {dm:+.3f} ({dci[0]:+.3f}..{dci[1]:+.3f}) -> {"HOLDS" if HH1c else "FAILS"}')
say(f'H-H2b combined (persistence + neighbours) minus persistence Spearman >= 0.02, lower bound > 0: {GAIN:+.3f} ({cg[0]:+.3f}..{cg[1]:+.3f}) -> {"HOLDS" if HH2b else "FAILS"}')
say(f'H-H5 lift of an Alert next hour when >= 3 of 5 neighbours were followed by an Alert >= 2.0 with lower bound > 1: {L5:.2f} ({c5[0]:.2f}..{c5[1]:.2f}) -> {"HOLDS" if HH5 else "FAILS"}')
say(f'H-H3 no duplicates, no leakage, median query <= 50 ms: {DUP}, {LEAK}, {LATMS:.1f} ms -> {"HOLDS" if HH3 else "FAILS"}')
say(f'H-H4 top-5 overlap under noise >= 0.60: {OV:.3f} -> {"HOLDS" if HH4 else "FAILS"}')
HYP = {'H-H1a': HH1a, 'H-H1c': HH1c, 'H-H2b': HH2b, 'H-H5': HH5, 'H-H3': HH3, 'H-H4': HH4}; say(f'{sum(HYP.values())} of 6 hold. Failed hypotheses are reported as failed; nothing was retuned. Retrieval stays historical CONTEXT, not a forecaster. Labels for H-H1 are synthetic; history is short.')
RES = {'protocol_h1_v4_sha256': PHH, 'chosen': CHOSEN, 'k_out': KBEST, 'index_versions': VERS, 'hypotheses': HYP, 'H1a': H1a, 'H1c': H1c_v, 'vs_score_only': [dm, [float(x) for x in dci]], 'vs_c1': [dm3, [float(x) for x in dci3]], 'gain_over_persistence': [GAIN, [float(x) for x in cg]], 'lift_alert_next_hour': [L5, [float(x) for x in c5]], 'persistence_lift': LPp, 'share_hours_mostly_alert': grp_share, 'duplicates': DUP, 'leakage': LEAK, 'median_query_ms': LATMS, 'noise_overlap': OV, 'mock': MOCK}
(OUT/'h1v4_results.json').write_text(json.dumps(RES, indent=1, default=str)); (OUT/'h1v4_share_block.txt').write_text('\n'.join(LINES))
say('Saved: h1v4_results.json, h1v4_selection_table.csv, h1v4_test3_table.csv, history_index_v2_*.npz, history_index_v2.json, h1v4_share_block.txt in models_b2_v1/ . Send the share block to Claude.')


=== pre-registered hypotheses (protocol H1 v4; new analysis on saved TEST-3 / TEST-B scores; v3 results are unchanged and stay as reported) ===
H-H1a precision@5 "event window" >= 0.50 (TEST-3, chosen c2): 0.543 -> HOLDS
H-H1c same-family P@5 >= 0.25 and >= score-only + 0.02 (paired lower bound > 0): 0.298; vs score-only +0.027 (+0.012..+0.041) -> HOLDS
H-H2b combined (persistence + neighbours) minus persistence Spearman >= 0.02, lower bound > 0: +0.052 (+0.030..+0.072) -> HOLDS
H-H5 lift of an Alert next hour when >= 3 of 5 neighbours were followed by an Alert >= 2.0 with lower bound > 1: 1.97 (0.00..6.19) -> FAILS
H-H3 no duplicates, no leakage, median query <= 50 ms: 0, 0, 13.7 ms -> HOLDS
H-H4 top-5 overlap under noise >= 0.60: 0.853 -> HOLDS
5 of 6 hold. Failed hypotheses are reported as failed; nothing was retuned. Retrieval stays historical CONTEXT, not a forecaster. Labels for H-H1 are synthetic; history is short.
Saved: h1v4_results.json, h1v4_selection_table.csv, h1v4_test3_